# Lesson 1.3 — What Makes a Trajectory 'Good'

*Module 7 · Unit 1*

**This notebook:** implements four quality predicates (continuity, limits, clearance, conditioning) and reproduces the accept/reject verdicts.

Run top to bottom (Restart & Run All). It ends by printing `All checks passed.`

In [ ]:
# --- Module 7 engine (embedded; builds on the M6 velocity layer) ---
"""
Module 7 engine - Trajectory Generation (Installment A: Units 1-2).
Builds ON the Module 6 velocity layer (imported verbatim, not reimplemented).
Greenhouse arm: planar 2R, L1=0.4, L2=0.3 (as in M5/M6); redundant 3R extension.
"""
import numpy as np

# ----- Module 6 base (reused verbatim from lesson32_capstone_velocity_layer) -----
def dh(th, d, a, al):
    ct, st, ca, sa = np.cos(th), np.sin(th), np.cos(al), np.sin(al)
    return np.array([[ct, -st*ca, st*sa, a*ct],
                     [st,  ct*ca, -ct*sa, a*st],
                     [0,      sa,     ca,   d ],
                     [0,       0,      0,   1 ]])

def forward_chain(P, T, q):
    M = np.eye(4); Ms = [M.copy()]
    for i, (th0, d0, a, al) in enumerate(P):
        th, d = (th0+q[i], d0) if T[i] == "R" else (th0, d0+q[i])
        M = M @ dh(th, d, a, al); Ms.append(M.copy())
    return Ms

def geometric_jacobian(P, T, q):
    Ms = forward_chain(P, T, q); on = Ms[-1][:3, 3]; J = np.zeros((6, len(q)))
    for i in range(len(q)):
        z = Ms[i][:3, 2]; o = Ms[i][:3, 3]
        if T[i] == "R": J[:3, i] = np.cross(z, on-o); J[3:, i] = z
        else: J[:3, i] = z
    return J

def Jv_planar(P, T, q): return geometric_jacobian(P, T, q)[:2, :]
def fk_xy(P, T, q): return forward_chain(P, T, q)[-1][:2, 3]

EPS = 0.08  # singularity threshold on sigma_min (M6 convention)

def analyze(P, T, q):
    J = Jv_planar(P, T, q); U, S, Vt = np.linalg.svd(J)
    w = float(np.prod(S)); kappa = float(S[0]/max(S[-1], 1e-12))
    return {"sigma": S, "w": w, "kappa": kappa,
            "axes": [(U[:, i], float(S[i])) for i in range(len(S))],
            "singular": bool(S[-1] < EPS), "sigma_min": float(S[-1]),
            "J": J, "U": U, "Vt": Vt}

def dls(J, lam): return J.T @ np.linalg.inv(J @ J.T + lam**2*np.eye(J.shape[0]))

def schedule_lambda(sigma_min, lam_max=0.1, eps=EPS):
    if sigma_min >= eps: return 0.0
    return float(np.sqrt((1-(sigma_min/eps)**2))*lam_max)

def velocity_layer(P, T, q, xi_d, z=None):
    """M6 deliverable: commanded tool twist -> joint rates (open-loop, singularity-robust)."""
    rep = analyze(P, T, q); lam = schedule_lambda(rep["sigma_min"])
    J = rep["J"]; Jd = dls(J, lam) if lam > 0 else np.linalg.pinv(J)
    qd = Jd @ xi_d
    if z is not None:
        qd = qd + (np.eye(len(q)) - Jd @ J) @ z
    info = {"w": rep["w"], "kappa": rep["kappa"], "sigma_min": rep["sigma_min"],
            "lambda": lam, "singular": rep["singular"]}
    return qd, info

# Canonical greenhouse arms
P2 = [(0, 0, 0.4, 0), (0, 0, 0.3, 0)]; T2 = ["R", "R"]                       # 2R, L1=0.4 L2=0.3
P3 = [(0, 0, 0.4, 0), (0, 0, 0.3, 0), (0, 0, 0.2, 0)]; T3 = ["R", "R", "R"]  # redundant 3R

# ----- Module 7 NEW: time parameterization (Units 1-2) -----
def poly_eval(c, t):
    """Evaluate ascending-coeff polynomial c=[c0..cn] and its 1st-3rd derivatives at t."""
    t = np.asarray(t, float); n = len(c)
    pos = sum(c[k]*t**k for k in range(n))
    vel = sum(k*c[k]*t**(k-1) for k in range(1, n)) if n > 1 else np.zeros_like(t)
    acc = sum(k*(k-1)*c[k]*t**(k-2) for k in range(2, n)) if n > 2 else np.zeros_like(t)
    jrk = sum(k*(k-1)*(k-2)*c[k]*t**(k-3) for k in range(3, n)) if n > 3 else np.zeros_like(t)
    return pos, vel, acc, jrk

def cubic_coeffs(q0, qf, v0, vf, T):
    """Cubic q(t) on [0,T] matching position+velocity endpoints. Ascending coeffs (C1)."""
    a0 = q0; a1 = v0
    a2 = (3*(qf-q0) - (2*v0+vf)*T) / T**2
    a3 = (2*(q0-qf) + (v0+vf)*T) / T**3
    return np.array([a0, a1, a2, a3])

def quintic_coeffs(q0, qf, v0, vf, a0, af, T):
    """Quintic q(t) on [0,T] matching position+velocity+acceleration endpoints. Ascending (C2)."""
    c0 = q0; c1 = v0; c2 = a0/2.0
    T2, T3, T4, T5 = T**2, T**3, T**4, T**5
    h = qf - q0
    c3 = (20*h - (8*vf + 12*v0)*T - (3*a0 - af)*T2) / (2*T3)
    c4 = (-30*h + (14*vf + 16*v0)*T + (3*a0 - 2*af)*T2) / (2*T4)
    c5 = (12*h - (6*vf + 6*v0)*T - (a0 - af)*T2) / (2*T5)
    return np.array([c0, c1, c2, c3, c4, c5])

def trapezoidal_profile(dist, v_max, a_max, n=400):
    """Trapezoidal velocity profile over scalar distance. Returns t, s, v, a, T_total.
    Acceleration is piecewise-constant -> discontinuous (jerk impulses at the corners)."""
    dist = float(dist); sgn = 1.0 if dist >= 0 else -1.0; D = abs(dist)
    if D == 0: t = np.linspace(0,1,n); z=np.zeros_like(t); return t,z,z,z,1.0
    t_acc = v_max / a_max
    d_acc = 0.5 * a_max * t_acc**2
    if 2*d_acc > D:                      # triangular (never reaches v_max)
        t_acc = np.sqrt(D / a_max); v_peak = a_max*t_acc; t_flat = 0.0
    else:
        v_peak = v_max; t_flat = (D - 2*d_acc) / v_max; d_acc = 0.5*a_max*t_acc**2
    Tf = 2*t_acc + t_flat
    t = np.linspace(0, Tf, n)
    s = np.zeros_like(t); v = np.zeros_like(t); a = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < t_acc:
            a[i] = a_max; v[i] = a_max*ti; s[i] = 0.5*a_max*ti**2
        elif ti < t_acc + t_flat:
            a[i] = 0.0; v[i] = v_peak; s[i] = d_acc + v_peak*(ti - t_acc)
        else:
            td = ti - t_acc - t_flat
            a[i] = -a_max; v[i] = v_peak - a_max*td
            s[i] = d_acc + v_peak*t_flat + v_peak*td - 0.5*a_max*td**2
    return t, sgn*s, sgn*v, sgn*a, Tf

def s_curve_profile(dist, v_max, a_max, j_max, n=600):
    """Jerk-limited (S-curve) profile by integrating a 7-segment jerk sequence.
    Returns t, s, v, a, j, T_total. Jerk is bounded by j_max -> acceleration is continuous."""
    dist = float(dist); sgn = 1.0 if dist >= 0 else -1.0; D = abs(dist)
    if D == 0: t = np.linspace(0,1,n); z=np.zeros_like(t); return t,z,z,z,z,1.0
    Tj = a_max / j_max
    t_const_a = max(v_max/a_max - Tj, 0.0)
    dt = 5e-4
    def simulate(coast):
        seg = [(+j_max, Tj), (0.0, t_const_a), (-j_max, Tj),
               (0.0, coast),
               (-j_max, Tj), (0.0, t_const_a), (+j_max, Tj)]
        ts=[0.0]; s=[0.0]; v=[0.0]; a=[0.0]; j=[0.0]
        for jv, dur in seg:
            steps = max(int(round(dur/dt)), 0)
            for _ in range(steps):
                a_n = a[-1]+jv*dt; v_n = v[-1]+a[-1]*dt; s_n = s[-1]+v[-1]*dt
                ts.append(ts[-1]+dt); j.append(jv); a.append(a_n); v.append(v_n); s.append(s_n)
        return (np.array(ts),np.array(s),np.array(v),np.array(a),np.array(j))
    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid=(lo+hi)/2; _,s,_,_,_ = simulate(mid)
        if s[-1] < D: lo=mid
        else: hi=mid
    ts,s,v,a,j = simulate((lo+hi)/2)
    tt=np.linspace(0,ts[-1],n)
    s=np.interp(tt,ts,s); v=np.interp(tt,ts,v); a=np.interp(tt,ts,a); j=np.interp(tt,ts,j)
    return tt, sgn*s, sgn*v, sgn*a, sgn*j, float(ts[-1])


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless; figures still render in Jupyter
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
checks = []

In [ ]:
def is_C2_ish(qdd, tol):  # flag large jumps between consecutive accel samples
    return bool(np.max(np.abs(np.diff(qdd))) < tol)
def within_limits(qd, qdd, vmax, amax):
    return bool(np.max(np.abs(qd))<=vmax+1e-9 and np.max(np.abs(qdd))<=amax+1e-9)
def worst_conditioning(qs):  # smallest sigma_min along a joint path (M6 analyze)
    return min(analyze(P2,T2,q)['sigma_min'] for q in qs)
# candidate D: smooth quintic move, within generous limits, well-conditioned
T=2.0; t=np.linspace(0,T,300); dt=t[1]-t[0]
c=quintic_coeffs(0.4,1.0,0,0,0,0,T); q,qd,qdd,_=poly_eval(c,t)
checks.append(is_C2_ish(qdd, 5.0))                       # quintic accel is smooth
checks.append(within_limits(qd,qdd, 5.0, 10.0))          # within generous limits
qs=np.stack([q, np.full_like(q,1.0)],axis=1)             # vary joint1, hold joint2
cond=worst_conditioning(qs); checks.append(cond>0)       # finite conditioning
print('candidate D: C2-ish=%s, within-limits=%s, worst sigma_min=%.3f'%(is_C2_ish(qdd,5.0), within_limits(qd,qdd,5.0,10.0), cond))
# candidate A: linear-in-time move has a VELOCITY JUMP at the ends (never eases from rest)
q_lin=np.linspace(0.4,1.0,300); qd_lin=np.gradient(q_lin,dt)
started_at_rest = abs(qd_lin[1])<1e-2
checks.append(not started_at_rest)        # linear does NOT start from rest -> C0 violation
print('candidate A (linear): starting speed=%.3f rad/s (nonzero -> velocity jump) -> rejected'%qd_lin[1])

In [ ]:
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')